# 01 — Generación de Tensores Multicanal
**EnergyFingerprint Pipeline** | Notebook 1 de 3

Genera tensores multicanal (9 o 11 canales) para variantes missense de un gen,
a partir de su CDS, variantes ClinVar, y opcionalmente scores ESM-1v.

**Prerrequisitos** (ejecutar como scripts `.py`):
- `01_download_<gen>.py` → `data/<gen>_clinvar_missense.csv` + `data/<gen>_cds.fasta`
- `03_run_esm.py` → `data/<gen>_esm_scores.csv` (opcional, para 11 canales)

**Salida**: `data/<gen>_tensors_<N>ch.npz`

## Configuración

In [ ]:
# ══════════════════════════════════════════════════
# CONFIGURAR AQUÍ — cambiar gen y subset para reproducir
# ══════════════════════════════════════════════════
GENE        = "scn1a"          # "brca1", "tp53", o nuevo gen
SUBSET      = "likely"         # "likely" (LP+LB) o "all" (P+B)
N_CHANNELS  = 11               # 9 (solo energía) o 11 (con ESM-1v)
WINDOW_SIZE = 128              # ventana en nucleótidos
# ══════════════════════════════════════════════════

## Imports y paths

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Añadir raíz del proyecto al path
PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

from energyfingerprint.energy import stacking_profile, STACKING_SANTALUCIA, STACKING_TURNER
from energyfingerprint.genetics import load_fasta, find_missense_codon, translate_cds

# Paths derivados del gen
STUDY_DIR = os.path.join(PROJECT_ROOT, "studies", GENE)
DATA_DIR  = os.path.join(STUDY_DIR, "data")
FIG_DIR   = os.path.join(STUDY_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

print(f"Gen: {GENE.upper()}")
print(f"Datos: {DATA_DIR}")

## 1. Cargar secuencia CDS

In [ ]:
cds_path = os.path.join(DATA_DIR, f"{GENE}_cds.fasta")
cds = load_fasta(cds_path)

protein = translate_cds(cds)
print(f"CDS: {len(cds)} nt → {len(protein)} aa")
print(f"Primeros 30 nt: {cds[:30]}...")
print(f"Primeros 10 aa: {protein[:10]}...")
assert cds[:3] == "ATG", "ERROR: CDS no comienza con ATG" 

## 2. Cargar variantes ClinVar

In [ ]:
csv_path = os.path.join(DATA_DIR, f"{GENE}_clinvar_missense.csv")
df = pd.read_csv(csv_path)

# Codificar labels
# IMPORTANTE: Excluir "Conflicting classifications of pathogenicity"
# porque contiene la subcadena "pathogenic" pero NO son variantes patogénicas.
# También excluir "risk factor", "drug response", etc.
def encode_label(sig):
    s = sig.lower().strip()
    # Excluir primero las categorías ambiguas/conflictivas
    if "conflicting" in s:
        return -1
    if "uncertain" in s:
        return -1
    if "not provided" in s:
        return -1
    if "risk factor" in s:
        return -1
    if "drug response" in s:
        return -1
    # Ahora sí clasificar
    if "pathogenic" in s: return 1
    if "benign" in s: return 0
    return -1

df["label"] = df["clinical_significance"].apply(encode_label)

# Mostrar distribución completa antes de filtrar
print("Distribución completa ClinVar:")
print(df.groupby("clinical_significance")["label"].agg(["count", "first"]).rename(
    columns={"count": "n", "first": "label_asignado"}).to_string())
print()

# Filtrar por subset
if SUBSET == "likely":
    df = df[df["clinical_significance"].str.contains("Likely", case=False, na=False)].copy()
    # Asegurar que solo quedan Likely pathogenic y Likely benign (no Conflicting)
    df = df[df["label"] >= 0].copy()
    subset_name = "Likely (LP + LB)"
else:
    df = df[df["label"] >= 0].copy()
    subset_name = "All (P + B)"

n_path = (df["label"] == 1).sum()
n_ben  = (df["label"] == 0).sum()
print(f"Subset: {subset_name}")
print(f"Variantes: {len(df)} ({n_path} P / {n_ben} B)")
print(f"\nDistribución clinical_significance (tras filtro):")
print(df["clinical_significance"].value_counts().to_string())

## 3. Cargar ESM-1v scores (opcional)

In [ ]:
esm_path = os.path.join(DATA_DIR, f"{GENE}_esm_scores.csv")
esm_map = {}

if os.path.exists(esm_path):
    df_esm = pd.read_csv(esm_path)
    # Mapear uid → esm_llr
    esm_map = dict(zip(df_esm["uid"], df_esm["esm_llr"]))
    has_esm = True
    print(f"ESM-1v scores cargados: {len(esm_map)} variantes")
    print(f"  |LLR| media: {df_esm['esm_llr'].abs().mean():.3f}")
    print(f"  |LLR| rango: [{df_esm['esm_llr'].min():.2f}, {df_esm['esm_llr'].max():.2f}]")
else:
    has_esm = False
    print("ESM-1v scores no encontrados → se usarán 9 canales")
    if N_CHANNELS >= 11:
        print("AVISO: Reduciendo a 9 canales (ejecuta 03_run_esm.py primero)")
        N_CHANNELS = 9

## 4. Funciones de generación de tensores

Cada variante se convierte en un tensor `(window_size, n_channels)`:

| Canal | Descripción | Fuente |
|-------|-------------|--------|
| C1-C2 | ΔG stacking WT (SantaLucia / Turner) | Nearest-neighbor |
| C3-C4 | ΔG stacking MUT (SantaLucia / Turner) | Nearest-neighbor |
| C5-C6 | ΔΔG = MUT - WT (SantaLucia / Turner) | Derivado |
| C7 | GC content (ventana 11nt) | Secuencia |
| C8 | Purine bias (ventana 11nt) | Secuencia |
| C9 | Posición en codón (0, 0.5, 1) | Secuencia |
| C10 | Estructura secundaria (placeholder) | — |
| C11 | ESM-1v LLR × gaussiana | ESM-1v |

In [ ]:
def compute_gc_content(seq, window=11):
    n = len(seq)
    gc = np.array([1.0 if c in ("G", "C") else 0.0 for c in seq.upper()])
    result = np.zeros(n)
    half = window // 2
    for i in range(n):
        lo, hi = max(0, i - half), min(n, i + half + 1)
        result[i] = gc[lo:hi].mean()
    return result

def compute_purine_bias(seq, window=11):
    n = len(seq)
    pur = np.array([1.0 if c in ("A", "G") else 0.0 for c in seq.upper()])
    result = np.zeros(n)
    half = window // 2
    for i in range(n):
        lo, hi = max(0, i - half), min(n, i + half + 1)
        result[i] = pur[lo:hi].mean()
    return result

def compute_codon_position(n):
    return np.array([{0: 0.0, 1: 0.5, 2: 1.0}[i % 3] for i in range(n)])

def extract_window(profile, center, ws=128):
    half = ws // 2
    n = len(profile)
    s, e = center - half, center + half
    w = np.zeros(ws)
    ss, se = max(0, s), min(n, e)
    ds = ss - s
    w[ds:ds + (se - ss)] = profile[ss:se]
    return w

def make_tensor(cds, aa_pos, alt_aa, esm_llr=0.0, n_channels=9, ws=128):
    """Genera tensor multicanal para una variante missense."""
    cs = (aa_pos - 1) * 3
    if cs + 3 > len(cds):
        return None
    wt_codon = cds[cs:cs + 3]
    mut_codon = find_missense_codon(wt_codon, alt_aa)
    if mut_codon is None:
        return None

    mut_cds = cds[:cs] + mut_codon + cds[cs + 3:]
    cp = cs + 1  # centro del codón

    channels = [
        extract_window(stacking_profile(cds, STACKING_SANTALUCIA), cp, ws),
        extract_window(stacking_profile(cds, STACKING_TURNER), cp, ws),
        extract_window(stacking_profile(mut_cds, STACKING_SANTALUCIA), cp, ws),
        extract_window(stacking_profile(mut_cds, STACKING_TURNER), cp, ws),
        extract_window(stacking_profile(mut_cds, STACKING_SANTALUCIA) -
                       stacking_profile(cds, STACKING_SANTALUCIA), cp, ws),
        extract_window(stacking_profile(mut_cds, STACKING_TURNER) -
                       stacking_profile(cds, STACKING_TURNER), cp, ws),
        extract_window(compute_gc_content(cds), cp, ws),
        extract_window(compute_purine_bias(cds), cp, ws),
        extract_window(compute_codon_position(len(cds)), cp, ws),
    ]

    if n_channels >= 10:
        channels.append(np.zeros(ws))  # SecStruct placeholder

    if n_channels >= 11:
        center = ws // 2
        sigma = ws / 6
        x = np.arange(ws)
        gaussian = np.exp(-0.5 * ((x - center) / sigma) ** 2)
        channels.append(esm_llr * gaussian)

    return np.stack(channels, axis=1)

## 5. Generar tensores para todas las variantes

In [ ]:
tensors_list, labels_list, uids_list = [], [], []
skipped = 0

for _, row in df.iterrows():
    aa_pos = int(row["aa_position"])
    alt_aa = row["alt_aa"]
    esm_llr = esm_map.get(row["uid"], 0.0)
    if pd.isna(esm_llr):
        esm_llr = 0.0

    t = make_tensor(cds, aa_pos, alt_aa, esm_llr=esm_llr,
                    n_channels=N_CHANNELS, ws=WINDOW_SIZE)
    if t is None:
        skipped += 1
        continue

    tensors_list.append(t)
    labels_list.append(row["label"])
    uids_list.append(row["uid"])

tensors = np.array(tensors_list, dtype=np.float32)
labels  = np.array(labels_list, dtype=np.int64)
uids    = np.array(uids_list)

print(f"Tensores generados: {tensors.shape}")
print(f"Omitidos: {skipped}")
print(f"Distribución: {(labels==1).sum()} P / {(labels==0).sum()} B")

## 6. Normalización Z-score por canal

In [ ]:
def normalize_tensors(t, ref_stats=None):
    """Z-score por canal. Opcionalmente usa estadísticas externas."""
    t_norm = t.copy()
    stats = {}
    for c in range(t.shape[2]):
        if ref_stats and c in ref_stats:
            mu, sigma = ref_stats[c]["mean"], ref_stats[c]["std"]
        else:
            d = t[:, :, c].flatten()
            mu, sigma = d.mean(), d.std()
        if sigma > 1e-10:
            t_norm[:, :, c] = (t_norm[:, :, c] - mu) / sigma
        stats[c] = {"mean": float(mu), "std": float(sigma)}
    return t_norm, stats

tensors_norm, norm_stats = normalize_tensors(tensors)
print("Estadísticas de normalización por canal:")
ch_names = ["ΔG_SL_wt", "ΔG_Tu_wt", "ΔG_SL_mut", "ΔG_Tu_mut",
            "ΔΔG_SL", "ΔΔG_Tu", "GC", "Purine", "CodonPos",
            "SecStruct", "ESM_LLR"]
for c in range(tensors.shape[2]):
    name = ch_names[c] if c < len(ch_names) else f"C{c}"
    print(f"  {name:12s}: μ={norm_stats[c]['mean']:+.4f}  σ={norm_stats[c]['std']:.4f}")

## 7. Visualización de ejemplo

In [ ]:
# Mostrar un tensor de ejemplo (primera variante patogénica)
idx_path = np.where(labels == 1)[0][0]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle(f"{GENE.upper()} — Tensor ejemplo (variante patogénica)", fontweight="bold")

# Raw
ax = axes[0]
im = ax.imshow(tensors[idx_path].T, aspect="auto", cmap="RdBu_r",
               vmin=-3, vmax=3)
ax.set_ylabel("Canal")
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(ch_names[:N_CHANNELS], fontsize=8)
ax.set_title("Tensor raw")
plt.colorbar(im, ax=ax, shrink=0.6)

# Normalized
ax = axes[1]
im = ax.imshow(tensors_norm[idx_path].T, aspect="auto", cmap="RdBu_r",
               vmin=-3, vmax=3)
ax.set_ylabel("Canal")
ax.set_xlabel("Posición (ventana)")
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(ch_names[:N_CHANNELS], fontsize=8)
ax.set_title("Tensor normalizado (Z-score)")
plt.colorbar(im, ax=ax, shrink=0.6)

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"tensor_example_{N_CHANNELS}ch.png"), dpi=150)
plt.show()

## 8. Guardar tensores

In [ ]:
output_path = os.path.join(DATA_DIR,
    f"{GENE}_tensors_{N_CHANNELS}ch{'_' + SUBSET if SUBSET != 'all' else ''}.npz")

np.savez_compressed(output_path,
    tensors=tensors_norm,
    labels=labels,
    variant_ids=uids,
    norm_stats=norm_stats,
    config={
        "gene": GENE,
        "subset": SUBSET,
        "n_channels": N_CHANNELS,
        "window_size": WINDOW_SIZE,
    }
)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Guardado: {output_path}")
print(f"Tamaño: {size_mb:.1f} MB")
print(f"Shape: {tensors_norm.shape}")
print(f"\n→ Usar este archivo en notebook 02_train_and_evaluate.ipynb")

## 9. Distribución de variantes por posición

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

pos_p = df[df["label"] == 1]["aa_position"]
pos_b = df[df["label"] == 0]["aa_position"]

ax.hist(pos_b, bins=50, alpha=0.6, color="#4CAF50", label=f"Benign (n={len(pos_b)})")
ax.hist(pos_p, bins=50, alpha=0.6, color="#F44336", label=f"Pathogenic (n={len(pos_p)})")
ax.set_xlabel("Posición aminoácido")
ax.set_ylabel("Frecuencia")
ax.set_title(f"{GENE.upper()} — Distribución de variantes missense ({subset_name})")
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"variant_distribution_{SUBSET}.png"), dpi=150)
plt.show()